<a href="https://colab.research.google.com/github/Taheri1400/machine-learning-projects/blob/main/fake_news_detection_solved.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<h1 align=center style="line-height:200%;font-family:vazir;color:#0099cc">
<font face="vazirmatn" color="#0099cc">
تشخیص اخبار دروغین
</font>
</h1>

<h2 align=right style="line-height:200%;font-family:vazir;color:#0099cc">
<font face="vazirmatn" color="#0099cc">
مقدمه و صورت مسئله
</font>
</h2>

<p dir=rtl style="direction: rtl; text-align: justify; line-height:200%; font-family:vazir; font-size:medium">
<font face="vazirmatn" size=3>
مطمئنا بارها شنیده‌اید که در فضای مجازی اخبار دروغ و شایعه بسیار رایج است. افرادی هستند که به هر دلیل اقدام به انتشار خبرهای دروغ می‌کنند. برخی این دروغ‌ها از جنس خبرهای سیاسی، اقتصادی یا حواشی راجع به زندگی افراد مشهور و سلبریتی‌هاست. اما به هر نحوی که باشد، دروغ رفتار خوبی نیست.
    <br>
    پس به عنوان یک مخاطب خوب است که بتوانیم تشخیص دهیم چه خبری حقیقت و چه خبری دروغ است. اگر از دید یک خبرگزاری هم به ماجرا نگاه کنیم، متوجه خواهیم شد که اصلا دوست نداریم یک خبر دروغ اقتصادی یا سیاسی از زبان ما منتشر شود؛ در غیر این صورت عواقب پیشبینی‌نشده‌ای در انتظارمان خواهد بود!
    <br>
    پس حال که به اهمیت و کاربرد تشخیص اخبار دروغین پی برده‌ایم، خوب است خودمان تلاش کنیم مدلی آموزش دهیم که خبرهای دروغ را از حقیقت جدا کنیم.
    <br>
    البته تشخیص این امر، وابسته به توانایی پردازش زبان طبیعی است و با دانش یادگیری ماشین کلاسیک، به سادگی نمی‌توان مسئله رو حل کرد. برای اینکه مسئله قابل حل باشد، بخش «پردازش زبان طبیعی» را خود ما انجام داده‌ایم و نتیجه آن‌ را در قالب بخشی از مجموعه‌داده در اختیار شما قرار داده‌ایم.
</font>
</p>

<h2 align=right style="line-height:200%;font-family:vazir;color:#0099cc">
<font face="vazirmatn" color="#0099cc">
وارد کردن کتابخانه‌های مورد نیاز
</font>
</h2>

<p dir=rtl style="direction: rtl; text-align: justify; line-height:200%; font-family:vazir; font-size:medium">
<font face="vazirmatn" size=3>
    ابتدا کتابخانه‌های مورد نیازتان را وارد کنید.
</font>
</p>

In [ ]:
import numpy as np
import pandas as pd
import scipy.sparse as ss

from sklearn.model_selection import train_test_split

from sklearn.svm import SVC
from sklearn.decomposition import PCA
from sklearn.metrics import classification_report, f1_score

<h2 align=right style="line-height:200%;font-family:vazir;color:#0099cc">
<font face="vazirmatn" color="#0099cc">
معرفی مجموعه داده
</font>
</h2>

<p dir=rtl style="direction: rtl; text-align: justify; line-height:200%; font-family:vazir; font-size:medium">
<font face="vazirmatn" size=3>
مجموعه‌داده این تمرین، مانند تمرین‌های قبلی به دو مجموعه آموزش و آزمون تقسیم‌بندی می‌شود. اما نکته حائز اهمیت این است که هر کدام از این دو مجموعه آزمون و آموزش، خود دارای دو مجموعه داده است. به عبارت بهتر مجموعه آموزش دارای دو مجموعه‌داده به اسم <code>news_train.csv</code> و <code>news_train_text_vectors.npz</code> است (مجموعه آزمون هم همینطور است).
    <br>
    در فایل <code>news_train.csv</code> اطلاعات کلی مانند زمان انتشار، منبع و برچسب خبر آمده است؛ اما خود خبر را در این مجموعه‌داده نداریم. بلکه خود خبر را می‌توان در فایل <code>news_train_text_vectors.npz</code> جستجو کرد. اما در <code>news_train_text_vectors.npz</code> متن خبر قرار نگرفته است، بلکه به کمک روش‌های پردازش زبان طبیعی، متن هر خبر را به یک بردار ۴۲۱۴۱ المانی تبدیل کرده‌ایم و به ازای هر خبر، یک بردار با ابعاد <code>(42141, 1)</code> وجود دارد.
    <br>
    توضیحات مجموعه‌داده <code>news_train.csv</code> را در جدول زیر می‌توانید مشاهده کنید.
</font>
</p>

<center>
<div dir=rtl style="direction: rtl;line-height:200%;font-family:vazir;font-size:medium">
<font face="vazirmatn" size=3>
    
|ستون|توضیحات|
|:------:|:---:|
|author|شخصی که متن خبر را نوشته است|
|published|تاریخ و ساعتی که خبر منتشر شده است|
|site_url|سایت منتشرکننده خبر|
|type|برای هر خبر یک دسته یا یک نوعی نیز ذخیره کرده‌ایم. این ستون می‌تواند یکی از مقادیر `bs` به معنی *پرت* یا ‍‍‍‍`conspiracy` به معنی *توطئه آمیز* یا `bias` به معنی *سوگیری یا متعصبانه* یا `hate` به معنی *نفرت‌افکن یا خشم‌آلود* یا ‍`satrie` به معنی *طنز* یا `junksci` به معنی *به درد نخور یا آشغال* و `fake` به معنی *دروغ* باشد|
|label|برچسب خبر را نشان می‌دهد. این ستون یا `Fake` است یا `Real`|
    
</font>
</div>
</center>


<p dir=rtl style="direction: rtl; text-align: justify; line-height:200%; font-family:vazir; font-size:medium">
<font face="vazirmatn" size=3>
    در کنار فایل <code>news_train.csv</code> فایل <code>news_train_text_vectors.npz</code> نیز قرار دارد. کلیات این فایل را در قسمت قبلی توضیح دادیم. به حجیم بودن فایل و صرفه‌جویی درفضای ذخیره‌سازی و کاهش زمان برای بارگذاری فایل، به جای فرمت <code>csv</code> آن را در فرمت <code>npz</code> در اختیار شما قرار دادیم. اما پیشنهاد می‌کنیم برای راحتی خودتان، پس از خواندن فایل، آن را به یک دیتافریم با ۱۵۰۰ سطر و ۴۲۱۴۱ ستون تبدیل کنید.
    <br>
    اینطور در نظر داشته باشید که متن هر خبر را با ۴۲۱۴۱ ویژگی برایتان توصیف کرده‌ایم!
</font>
</p>


<h2 align=right style="line-height:200%;font-family:vazir;color:#0099cc">
<font face="vazirmatn" color="#0099cc">
خواندن مجموعه داده
</font>
</h2>

<p dir=rtl style="direction: rtl; text-align: justify; line-height:200%; font-family:vazir; font-size:medium">
<font face="vazirmatn" size=3>
    در ابتدا نیاز است فایل‌های مجموعه‌داده را بخوانید. نمونه‌های آموزشی در فایل <code>news_train.csv</code> و <code>news_train_text_vectors.npz</code> و نمونه‌های آزمون که باید دسته‌ی آن‌ها را پیش‌بینی کنید در فایل <code>news_test.csv</code> و <code>news_test_text_vectors.npz</code> ذخیره شده‌اند. اگر لازم دانستید می‌توانید به دلخواه خود بخشی از دادگان آموزشی را به عنوان دادگان اعتبارسنجی نیز جدا کنید.
</font>
</p>

In [ ]:
train_data = pd.read_csv('news_train.csv')
train_data_text_vectors = ss.load_npz('news_train_text_vectors.npz')

test_data = pd.read_csv('news_test.csv')
test_data_text_vectors = ss.load_npz('news_test_text_vectors.npz')

In [ ]:
train_data

,author,published,site_url,type,label
0,No Author,2016-11-01T03:28:50.389+02:00,clickhole.com,satire,Fake
1,Anonymous,2016-10-27T21:30:00.000+03:00,abeldanger.net,bs,Fake
2,Kali74,2016-10-27T02:54:49.093+03:00,abovetopsecret.com,bs,Fake
3,Alex Ansary,2016-11-04T22:44:06.026+02:00,amtvmedia.com,bs,Fake
4,Luke Stranahan,2016-11-23T15:10:56.702+02:00,returnofkings.com,hate,Real
...,...,...,...,...,...
1495,Daniel Greenfield,2016-11-02T13:25:57.336+02:00,frontpagemag.com,hate,Real
1496,theeconomiccollapseblog.com,http://theeconomiccollapseblog.com/wp-content/...,theeconomiccollapseblog.com,bs,Fake
1497,Alex Ansary,2016-11-02T19:24:00.901+02:00,amtvmedia.com,bs,Fake
1498,EdJenner,2016-11-22T15:44:03.678+02:00,dailywire.com,bias,Real


In [ ]:
train_data_text_vectors

<COOrdinate sparse matrix of dtype 'float64'
	with 284524 stored elements and shape (1500, 42141)>

<h2 align=right style="line-height:200%;font-family:vazir;color:#0099cc">
<font face="vazirmatn" color="#0099cc">
پیش‌پردازش و مهندسی ویژگی
</font>
</h2>

<p dir=rtl style="direction: rtl; text-align: justify; line-height:200%; font-family:vazir; font-size:medium">
<font face="vazirmatn" size=3>
    در این سوال شما می‌توانید از هر تکنیک پیش‌پردازش/مهندسی ویژگی که در فصل‌های گذشته آموختید، استفاده کنید.
    <br>
    تکنیک‌هایی که استفاده می‌کنید به شکل مستقیم مورد ارزیابی توسط سامانه داوری قرار <b>نمی‌گیرند.</b> بلکه همه آن‌ها در دقت مدل شما تاثیر خواهند گذاشت؛ بنابراین هر چه پیش‌پردازش/مهندسی ویژگی بهتری انجام دهید تا دقت مدل بهبود پیدا کند، امتیاز بیشتری از این سوال کسب خواهید کرد.
    <br>
    از آنجایی که قرار است در این تمرین از الگوریتم ماشین بردار پشتیبان استفاده کنید، حتما باید مجموعه‌داده <code>news_train_text_vectors.npz</code> را با روش‌های کاهش ابعادی که یاد گرفته‌اید، به نحوی تغییر دهید که در زمان معقولی توسط الگوریتم <code>svm</code> پردازش شود. اگر چه الگوریتمی که برای کاهش ابعاد استفاده می‌کنید در کوئرا بررسی نمی‌شود، اما پیشنهاد ما استفاده از <code>PCA</code> است. آنچه اهمیت دارد آن است که بین کوچک‌کردن داده و کیفیت مدل تعادل برقرار کنید. طبیعی است که هرچقدر داده کوچک‌تر باشد، احتمالا الگوریتم ماشین بردار پشتیبان زودتر به نتیجه می‌رسد؛ اما در نظر داشته باشید با کاهش بعد، شما عملا بخشی از داده را از دست می‌دهید!
</font>
</p>

In [ ]:
import pandas as pd
import scipy.sparse as ss
from sklearn.decomposition import PCA
from sklearn.preprocessing import LabelEncoder

# 1. بارگذاری داده‌ها
train_meta = pd.read_csv('news_train.csv')
train_vectors_sparse = ss.load_npz('news_train_text_vectors.npz')

# تبدیل sparse matrix به dense (برای PCA باید dense باشه)
train_vectors = train_vectors_sparse.toarray()

# 2. تعریف تابع برای تشخیص سایت‌های مشکوک، نویسندگان مشکوک، نوع خبر مشکوک
def detect_suspicious_entities(df, column_name, label_col='label', thresh=0.5):
    # تبدیل برچسب به عدد
    df_numeric = df.copy()
    df_numeric[label_col] = df_numeric[label_col].map({'Real':0, 'Fake':1})

    mean_fake_ratio = df_numeric.groupby(column_name)[label_col].mean()
    suspicious = mean_fake_ratio[mean_fake_ratio >= thresh].index.tolist()
    not_suspicious = mean_fake_ratio[mean_fake_ratio < thresh].index.tolist()
    return suspicious, not_suspicious

# سایت‌های مشکوک
suspicious_sites, _ = detect_suspicious_entities(train_meta, 'site_url')
# نویسندگان مشکوک
suspicious_authors, _ = detect_suspicious_entities(train_meta, 'author')
# نوع خبر مشکوک
suspicious_types, _ = detect_suspicious_entities(train_meta, 'type')

# 3. اضافه کردن ستون‌های باینری برای سایت/نویسنده/نوع مشکوک
train_meta['is_suspicious_site'] = train_meta['site_url'].apply(lambda x: 1 if x in suspicious_sites else 0)
train_meta['is_suspicious_author'] = train_meta['author'].apply(lambda x: 1 if x in suspicious_authors else 0)
train_meta['is_suspicious_type'] = train_meta['type'].apply(lambda x: 1 if x in suspicious_types else 0)

# 4. حذف ستون‌های اضافی (مثلا site_url، author، type، published)
train_meta = train_meta.drop(columns=['site_url', 'author', 'type', 'published'])

# 5. تبدیل برچسب به عدد (0 و 1)
train_meta['label'] = train_meta['label'].map({'Real':0, 'Fake':1})

# 6. کاهش ابعاد بردارهای متن با PCA به ابعاد 500
pca = PCA(n_components=500, random_state=42)
reduced_text_vectors = pca.fit_transform(train_vectors)

# 7. ترکیب داده‌های متا (جدول) با بردارهای متنی کاهش‌یافته
import numpy as np

# تبدیل dataframe متا به numpy array
meta_features = train_meta.drop(columns=['label']).values

# ادغام افقی ویژگی‌های متا و متن (np.hstack)
X = np.hstack([meta_features, reduced_text_vectors])

# 8. برچسب هدف
y = train_meta['label'].values

print('Shape of combined features:', X.shape)
print('Shape of labels:', y.shape)


Shape of combined features: (1500, 503)
Shape of labels: (1500,)


<h3 align=right style="direction: rtl;text-align: right;line-height:200%;font-family:vazir;color:#0099cc">
<font face="vazirmatn" color="#0099cc">
    استفاده از <code>scikit-learn</code>
</font>
</h3>


<p dir=rtl style="direction: rtl;text-align: right;line-height:200%;font-family:vazir;font-size:medium">
<font face="vazirmatn" size=3>
    الگوریتم <i>تحلیل مولفه‌های اصلی</i> با نام <code>PCA</code> در پکیج <code>decomposition</code> این کتابخانه در دسترس است. برخی از آرگومان‌های مهم آن در جدول زیر آمده است، اما جهت مطالعه‌ی کامل‌تر مستندات می‌توانید به <a href="https://scikit-learn.org/stable/modules/generated/sklearn.decomposition.PCA.html" target="_blank">این لینک</a> مراجعه فرمایید.
    <br>
    برای استفاده از این الگوریتم، می‌توانید با اجرای <code>from sklearn.decomposition import PCA</code> آن را وارد (<code>import</code>) کنید.
    <br>
    
</font>
</p>


<center>
<div dir=rtl style="direction: rtl;line-height:200%;font-family:vazir;font-size:medium">
<font face="vazirmatn" size=3>

|آرگومان|جنس و تایپ|توضیحات|
|:------:|:--------:|:---:|
|n_components|<code>int</code> یا <code>float</code>|تعداد مولفه‌هایی است که قصد داریم پس از اجرای الگوریتم، آن‌ها را داشته باشیم و حذف نشوند. اگر به این آرگومان مقدار ندهیم، همه مولفه‌ها حفظ خواهند شد. اگر جنس این آرگومان <code>int</code> باشد، نشانگر تعداد مولفه‌هایی است که قصد داریم حفظ شود. اما اگر از جنس <code>float</code> و به شکل $0< n\_components< 1$ باشد، به این معنا خواهد بود که به اندازه‌ای از مولفه‌ها حفظ شوند که واریانس داده‌ها بزرگتر از `n_componets`٪ باشد. بنابراین می‌توان اینگونه برداشت کرد که هرچه این عدد به ۱ نزدیک‌تر باشد، چون قرار است داده واریانس زیادی داشته باشد، پس مولفه‌های بیشتری حفظ می‌شوند.|
    
</font>
</div>
</center>



<p dir=rtl style="direction: rtl;text-align: right;line-height:200%;font-family:vazir;font-size:medium">
<font face="vazirmatn" size=3>
    کلاس <code>PCA</code> دارای سه متد به شرح زیر است:
</font>
</p>


<center>
<div dir=rtl style="direction: rtl;text-align: right;line-height:200%;font-family:vazir;font-size:medium">
<font face="vazirmatn" size=3>
    
|اسم متد|توضیحات|
|:------:|:---:|
|fit(X)|الگوریتم را با X که در ابعاد `(n_samples, n_features)` است|
|transform(X)|این تابع X که از ابعاد `(n_samples, n_features)` است را به یک ماتریس دیگری با ابعاد `(n_samples, n_components)` تبدیل می‌کند. پس خروجی این تابع یک ماتریس به ابعاد `(n_samples, n_components)` خواهد بود|
|fit_transform(X)|ابتدا تابع <code>fit</code> و سپس تابع <code>transform</code> را اجرا می‌کند|
</font>
</div>
</center>

In [ ]:
from sklearn.decomposition import PCA
import numpy as np

# فرض کنیم داده‌های ورودی X به شکل (تعداد نمونه‌ها، تعداد ویژگی‌ها)
X = np.random.rand(100, 1000)  # داده فرضی با 100 نمونه و 1000 ویژگی

# تعریف شی PCA با تعداد مولفه دلخواه، مثلا 50
pca = PCA(n_components=50)

# آموزش مدل PCA روی داده X و تبدیل آن به فضای جدید
X_reduced = pca.fit_transform(X)

print("Shape before PCA:", X.shape)        # (100, 1000)
print("Shape after PCA:", X_reduced.shape) # (100, 50)



Shape before PCA: (100, 1000)
Shape after PCA: (100, 50)


<h2 align=right style="line-height:200%;font-family:vazir;color:#0099cc">
<font face="vazirmatn" color="#0099cc">
مدل‌سازی
</font>
</h2>

<p dir=rtl style="direction: rtl; text-align: justify; line-height:200%; font-family:vazir; font-size:medium">
<font face="vazirmatn" size=3>
    حال که داده را پاکسازی کرده و احتمالا ویژگی‌هایی را به آن افزوده یا از آن حذف کرده‌اید، وقت آن است که مدلی آموزش دهید که بتواند متغیر هدف این مسئله را پیش‌بینی کند.
    <br>
    <b>توجه داشته باشید که برای پیشبینی متغیر هدف این مسئله حتما باید از الگوریتم <code>SVM</code> استفاده کنید. در غیر این صورت، نمره‌ای دریافت نخواهید کرد.</b>
    <br>
    نحوه استفاده از این الگوریتم را در تمرین‌ها و درسنامه‌های قبلی آموخته‌اید.
</font>
</p>

<h3 align=right style="line-height:200%;font-family:vazir;color:#0099cc">
<font face="vazirmatn" color="#0099cc">
آموزش مدل
</font>
</h3>

In [ ]:
from sklearn.svm import SVC
import pandas as pd
import scipy.sparse as ss
from sklearn.decomposition import PCA

# -- فرض: داده‌های پیش‌پردازش شده آماده است --
# train_meta : دیتافریم متادیتا با ستون label (عددی 0 و 1)
# train_vectors_sparse : sparse matrix بردارهای متن
# train_vectors_dense : ماتریس dense از بردارهای متن

# 1. کاهش بعد بردارهای متن
pca = PCA(n_components=500, random_state=42)
train_vectors_dense = train_vectors_sparse.toarray()  # تبدیل به dense برای PCA
reduced_train_text_vectors = pca.fit_transform(train_vectors_dense)

# 2. آموزش مدل SVC فقط روی بردارهای متن کاهش‌یافته
text_model = SVC()
text_model.fit(reduced_train_text_vectors, train_meta['label'])

# 3. اضافه کردن پیش‌بینی مدل متنی به دیتافریم متا
train_meta['label_from_text_model'] = text_model.predict(reduced_train_text_vectors)

# 4. حالا مدل نهایی را با داده‌های ترکیبی آموزش می‌دهیم
# فرض می‌کنیم در train_meta ستون‌های متادیتا (features) به جز 'label' و 'label_from_text_model' هست

# جدا کردن ویژگی‌ها (همه به جز label)
features = train_meta.drop(columns=['label'])

# آموزش مدل نهایی
model = SVC()
model.fit(features, train_meta['label'])

# اگر بخوای می‌تونی accuracy یا گزارش مدل رو هم اینجا محاسبه کنی
from sklearn.metrics import accuracy_score

# پیش‌بینی روی داده آموزش با مدل نهایی
train_pred = model.predict(features)
acc = accuracy_score(train_meta['label'], train_pred)
print(f"Training Accuracy of final model: {acc:.4f}")



Training Accuracy of final model: 1.0000


<h3 align=right style="line-height:200%;font-family:vazir;color:#0099cc">
<font face="vazirmatn" color="#0099cc">
معیار ارزیابی
</font>
</h3>

<p dir=rtl style="direction: rtl; text-align: justify; line-height:200%; font-family:vazir; font-size:medium">
<font face="vazirmatn" size=3>
    معیاری که برای ارزیابی عملکرد مدل انتخاب کرده‌ایم، <code>f1_score</code> نام دارد.
    <br>
    این معیار، سنجه ارزیابی کیفیت مدل شماست. به عبارت بهتر در سامانه داوری هم از همین معیار برای نمره‌دهی استفاده شده است.
    <br>
    پیشنهاد می‌شود با توجه به این معیار، عملکرد مدل خود را بر روی دادگان آموزش یا اعتبارسنجی ارزیابی کنید.
</font>
</p>

<p dir=rtl style="direction: rtl; text-align: justify; line-height:200%; font-family:vazir; font-size:medium">
<font color="red"><b color='red'>توجه:</b></font>
<font face="vazirmatn" size=3>
 جهت کسب امتیاز کامل نیاز است تا پاسخ شما حداقل امتیاز <code>80</code> را با توجه به معیار معرفی‌شده کسب نماید.
</font>
</p>

In [ ]:
from sklearn.metrics import f1_score

# فرض کنیم train_meta['label'] برچسب‌های اصلی هست
# train_pred هم پیش‌بینی مدل نهایی روی داده آموزش

f1 = f1_score(train_meta['label'], train_pred)
print(f"F1 Score on training data: {f1:.4f}")


F1 Score on training data: 1.0000


<h2 align=right style="line-height:200%;font-family:vazir;color:#0099cc">
<font face="vazirmatn" color="#0099cc">
 پیش‌بینی برای داده تست و خروجی
</font>
</h2>

<p dir=rtl style="direction: rtl;text-align: right;line-height:200%;font-family:vazir;font-size:medium">
<font face="vazirmatn" size=3>
    پس از مهندسی ویژگی و مدلسازی، الگوریتمی دارید که می‌تواند شما را از متغیرهای مستقل به متغیر هدف برساند.
    <br>
    از این مدل برای پیش‌بینی نمونه‌های موجود در داده تست استفاده کنید و نتایج را در قالب جدول (<code>dataframe</code>) زیر آماده کنید.
</font>
</p>

<div dir=rtl style="direction: rtl;text-align: right;line-height:200%;font-family:vazir;font-size:medium">
<font face="vazirmatn" size=3>
    
|ستون|توضیحات|
|------|---|
|label|برچسب پیشبینی شده|
    
</font>
</div>



<p dir=rtl style="direction: rtl;text-align: right;line-height:200%;font-family:vazir;font-size:medium">
<font face="vazirmatn" size=3>
    اسم دیتافریم باید <i>submission</i> باشد؛ در غیر این صورت، سامانه داوری نمی‌تواند تلاش‌ شما را ارزیابی کند.
    <br>
    این دیتافریم تنها شامل ۱ ستون با اسم <i>label</i> است و ۳۴۶ سطر دارد.
    <br>
    به ازای هر سطر موجود در دیتافریم <i>test</i> شما باید یک مقدار پیشبینی شده داشته باشید.
    <br>
    جدول زیر، ۵ سطر ابتدایی دیتافریم <code>submission</code> را نشان می‌دهد. البته در جواب شما، اعداد ستون <i>label</i> ممکن است متفاوت باشد.
</font>
</p>

<div style="text-align: right;line-height:200%;font-family:vazir;font-size:medium">
<font face="vazirmatn" size=3>
    
||label|
|----|-----|
|0|Fake|
|1|Fake|
|2|Real|
|3|Fake|
|4|Real|
</font>
</div>



In [ ]:
import pandas as pd
import scipy.sparse as ss

# بارگذاری داده تست
test_meta = pd.read_csv('news_test.csv')
test_vectors_sparse = ss.load_npz('news_test_text_vectors.npz')
test_vectors = test_vectors_sparse.toarray()

# --- پیش‌پردازش متادیتا تست مشابه train ---
# فرض می‌کنیم suspicious lists (suspicious_sites, suspicious_authors, suspicious_types) از قبل آماده است

test_meta['is_suspicious_site'] = test_meta['site_url'].apply(lambda x: 1 if x in suspicious_sites else 0)
test_meta['is_suspicious_author'] = test_meta['author'].apply(lambda x: 1 if x in suspicious_authors else 0)
test_meta['is_suspicious_type'] = test_meta['type'].apply(lambda x: 1 if x in suspicious_types else 0)

# حذف ستون‌های اضافی
test_meta = test_meta.drop(columns=['site_url', 'author', 'type', 'published'])

# کاهش ابعاد بردار متن تست با PCA که روی train فیت شده
reduced_test_text_vectors = pca.transform(test_vectors)

# پیش‌بینی مدل متن روی تست
test_meta['label_from_text_model'] = text_model.predict(reduced_test_text_vectors)

# آماده‌سازی ویژگی‌های تست (متادیتا + پیش‌بینی مدل متن)
X_test = test_meta.values

# پیش‌بینی نهایی مدل ترکیبی
test_pred = model.predict(X_test)

# برگرداندن پیش‌بینی عددی به رشته‌ای (0 -> Real، 1 -> Fake)
label_map = {0: 'Real', 1: 'Fake'}
test_pred_labels = [label_map[p] for p in test_pred]

# ساخت دیتافریم submission
submission = pd.DataFrame({'label': test_pred_labels})

submission



/usr/local/lib/python3.11/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but SVC was fitted with feature names
  warnings.warn(


,label
0,Real
1,Fake
2,Real
3,Real
4,Real
...,...
341,Fake
342,Real
343,Fake
344,Fake


<h2 align=right style="line-height:200%;font-family:vazir;color:#0099cc">
<font face="vazirmatn" color="#0099cc">
<b>سلول جواب‌ساز</b>
</font>
</h2>

<p dir=rtl style="direction: rtl; text-align: justify; line-height:200%; font-family:vazir; font-size:medium">
<font face="vazirmatn" size=3>
    برای ساخته‌شدن فایل <code>result.zip</code> سلول زیر را اجرا کنید. توجه داشته باشید که پیش از اجرای سلول زیر تغییرات اعمال شده در نت‌بوک را ذخیره کرده باشید (<code>ctrl+s</code>) تا در صورت نیاز به پشتیبانی امکان بررسی کد شما وجود داشته باشد.
</font>
</p>

In [ ]:
import zipfile
import joblib

def compress(file_names):
    print("File Paths:")
    print(file_names)
    compression = zipfile.ZIP_DEFLATED
    with zipfile.ZipFile("result.zip", mode="w") as zf:
        for file_name in file_names:
            zf.write('./' + file_name, file_name, compress_type=compression)

joblib.dump(model, 'model')
submission.to_csv('submission.csv', index=False)
file_names = ['fake_news_detection.ipynb', 'submission.csv', 'model']
compress(file_names)

File Paths:
['fake_news_detection.ipynb', 'submission.csv', 'model']
